# Stage 2 joint experiment notebook

Run order after Exp2 refactor: 00 prepare dataset → 01 Exp2A baseline → 02 Exp2B GCA baseline → 03 Exp2D lane detail neck → 04 Exp2E matched lane loss → 05 Exp2F detail + matching. Run 06 KD only if the teacher checkpoint exists. Run Exp3 notebooks only after Exp2F improves lane geometry. Every notebook re-extracts the Drive tar into `/content`; never assume files from a previous Colab runtime still exist. Training logs are printed in this notebook cell and mirrored to `/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/`.


# Stage 2 Notebook 08 - Joint evaluation, visualization, and video/GPU profile

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)


Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [ ]:
from pathlib import Path
import os, subprocess, sys, traceback

# Eval is read-only over saved tars. Each item is wrapped so one failure
# (missing tar, mismatched checkpoint, etc.) does not block the trend plot
# and video cells below. Exp2G is included so a fresh CLRKDLaneHead run is
# evaluated automatically once its tar shows up on Drive.
EVAL_ITEMS = [
    {
        'config': 'stage2/configs/exp01_rmt_shared_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp01_rmt_shared_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp02_rmt_gca_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp02_rmt_gca_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp03_rmt_gca_lane_detail_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp03_rmt_gca_lane_detail_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp04_rmt_gca_lane_matching_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp04_rmt_gca_lane_matching_joint_short10.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp04_rmt_gca_lane_matching_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp04_rmt_gca_lane_matching_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp05_rmt_gca_lane_detail_matching_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp05_rmt_gca_lane_detail_matching_joint_short10.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp05_rmt_gca_lane_detail_matching_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp05_rmt_gca_lane_detail_matching_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp07_rmt_gca_clrkd_lane_head_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp07_rmt_gca_clrkd_lane_head_joint_short10.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp07_rmt_gca_clrkd_lane_head_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp07_rmt_gca_clrkd_lane_head_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp08_rmt_gca_clrkd_asl_cls_rescue_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp08_rmt_gca_clrkd_asl_cls_rescue_joint_short10.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp08_rmt_gca_clrkd_asl_cls_rescue_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp08_rmt_gca_clrkd_asl_cls_rescue_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp09_rmt_gca_clrkd_ohem_prior_encoder_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp09_rmt_gca_clrkd_ohem_prior_encoder_joint_short10.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp09_rmt_gca_clrkd_ohem_prior_encoder_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp09_rmt_gca_clrkd_ohem_prior_encoder_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp10_rmt_gca_clrkd_separate_cls_path_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp10_rmt_gca_clrkd_separate_cls_path_joint_short10.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp10_rmt_gca_clrkd_separate_cls_path_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp10_rmt_gca_clrkd_separate_cls_path_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp11_rmt_gca_clrkd_iou_regression_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp11_rmt_gca_clrkd_iou_regression_joint_short10.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp11_rmt_gca_clrkd_iou_regression_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp11_rmt_gca_clrkd_iou_regression_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp12_rmt_gca_clrkd_iou_qfl_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp12_rmt_gca_clrkd_iou_qfl_joint_short10.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp12_rmt_gca_clrkd_iou_qfl_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp12_rmt_gca_clrkd_iou_qfl_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp13_rmt_gca_clrkd_iou_sqrt_target_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp13_rmt_gca_clrkd_iou_sqrt_target_joint_short10.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp13_rmt_gca_clrkd_iou_sqrt_target_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp13_rmt_gca_clrkd_iou_sqrt_target_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp14_rmt_gca_clrkd_decoded_eval_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp14_rmt_gca_clrkd_decoded_eval_joint_short10.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp14_rmt_gca_clrkd_decoded_eval_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp14_rmt_gca_clrkd_decoded_eval_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp15_rmt_gca_clrkd_oracle_diagnostic_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp15_rmt_gca_clrkd_oracle_diagnostic_joint_short10.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp15_rmt_gca_clrkd_oracle_diagnostic_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp15_rmt_gca_clrkd_oracle_diagnostic_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp16_rmt_gca_lane_query_head_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp16_rmt_gca_lane_query_head_joint_short10.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp16_rmt_gca_lane_query_head_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp16_rmt_gca_lane_query_head_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp17_rmt_gca_hybrid_prior_query_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp17_rmt_gca_hybrid_prior_query_joint_short10.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp17_rmt_gca_hybrid_prior_query_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp17_rmt_gca_hybrid_prior_query_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp18_rmt_gca_query_anchor_dab_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp18_rmt_gca_query_anchor_dab_joint_short10.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp18_rmt_gca_query_anchor_dab_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp18_rmt_gca_query_anchor_dab_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp19_rmt_gca_bezier_query_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp19_rmt_gca_bezier_query_joint_short10.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp19_rmt_gca_bezier_query_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp19_rmt_gca_bezier_query_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp20_rmt_gca_hybrid_with_stage1_aux_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp20_rmt_gca_hybrid_with_stage1_aux_joint_short10.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp20_rmt_gca_hybrid_with_stage1_aux_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp20_rmt_gca_hybrid_with_stage1_aux_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp21_rmt_gca_lane_only_diagnostic_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp21_rmt_gca_lane_only_diagnostic_joint_short10.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp21_rmt_gca_lane_only_diagnostic_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp21_rmt_gca_lane_only_diagnostic_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp22_rmt_gca_grid_det_with_query_lane_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp22_rmt_gca_grid_det_with_query_lane_joint_short10.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp22_rmt_gca_grid_det_with_query_lane_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp22_rmt_gca_grid_det_with_query_lane_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp23_rmt_gca_query_with_mask_aux_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp23_rmt_gca_query_with_mask_aux_joint_short10.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp23_rmt_gca_query_with_mask_aux_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp23_rmt_gca_query_with_mask_aux_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp24_rmt_gca_hybrid_combined_signals_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp24_rmt_gca_hybrid_combined_signals_joint_short10.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp24_rmt_gca_hybrid_combined_signals_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp24_rmt_gca_hybrid_combined_signals_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp25_rmt_gca_mask_fixed_lambda_long_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp25_rmt_gca_mask_fixed_lambda_long_joint_short15.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp25_rmt_gca_mask_fixed_lambda_long_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp25_rmt_gca_mask_fixed_lambda_long_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp26_rmt_gca_mask_uncertainty_weighting_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp26_rmt_gca_mask_uncertainty_weighting_joint_short15.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp26_rmt_gca_mask_uncertainty_weighting_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp26_rmt_gca_mask_uncertainty_weighting_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp27_rmt_gca_mask_high_resolution_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp27_rmt_gca_mask_high_resolution_joint_short15.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp27_rmt_gca_mask_high_resolution_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp27_rmt_gca_mask_high_resolution_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp28_rmt_gca_mask_lane_dominant_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp28_rmt_gca_mask_lane_dominant_joint_short15.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp28_rmt_gca_mask_lane_dominant_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp28_rmt_gca_mask_lane_dominant_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp29_rmt_gca_mask_uncertainty_long30_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp29_rmt_gca_mask_uncertainty_long30_joint_long30.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp29_rmt_gca_mask_uncertainty_long30_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp29_rmt_gca_mask_uncertainty_long30_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp30_rmt_gca_mask_self_distillation_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp30_rmt_gca_mask_self_distillation_joint_short15.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp30_rmt_gca_mask_self_distillation_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp30_rmt_gca_mask_self_distillation_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp31_rmt_gca_mask_uncertainty_cosine_lr_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp31_rmt_gca_mask_uncertainty_cosine_lr_joint_short20.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp31_rmt_gca_mask_uncertainty_cosine_lr_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp31_rmt_gca_mask_uncertainty_cosine_lr_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp32_rmt_gca_mask_self_distillation_cosine_lr_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp32_rmt_gca_mask_self_distillation_cosine_lr_joint_short20.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp32_rmt_gca_mask_self_distillation_cosine_lr_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp32_rmt_gca_mask_self_distillation_cosine_lr_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp33_rmt_gca_mask_uncertainty_full_dataset_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp33_rmt_gca_mask_uncertainty_full_dataset_joint_full15.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp33_rmt_gca_mask_uncertainty_full_dataset_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp33_rmt_gca_mask_uncertainty_full_dataset_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp34_rmt_gca_anchor_clrkd_modern_training_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp34_rmt_gca_anchor_clrkd_modern_training_joint_short20.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp34_rmt_gca_anchor_clrkd_modern_training_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp34_rmt_gca_anchor_clrkd_modern_training_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp35_rmt_gca_anchor_asl_amp_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp35_rmt_gca_anchor_asl_amp_joint_short20.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp35_rmt_gca_anchor_asl_amp_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp35_rmt_gca_anchor_asl_amp_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp36_rmt_gca_anchor_iou_regression_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp36_rmt_gca_anchor_iou_regression_joint_short20.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp36_rmt_gca_anchor_iou_regression_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp36_rmt_gca_anchor_iou_regression_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp37_rmt_gca_anchor_dual_score_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp37_rmt_gca_anchor_dual_score_joint_short20.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp37_rmt_gca_anchor_dual_score_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp37_rmt_gca_anchor_dual_score_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp38_rmt_gca_anchor_mask_consistency_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp38_rmt_gca_anchor_mask_consistency_joint_short20.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp38_rmt_gca_anchor_mask_consistency_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp38_rmt_gca_anchor_mask_consistency_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp39_rmt_gca_anchor_hungarian_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp39_rmt_gca_anchor_hungarian_joint_short20.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp39_rmt_gca_anchor_hungarian_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp39_rmt_gca_anchor_hungarian_joint.tar',
        ],
    },
    {
        'config': 'stage2/configs/exp40_rmt_gca_anchor_width1_long30_joint.yaml',
        'candidates': [
            '/content/drive/MyDrive/EcoCAR/training_runs/exp40_rmt_gca_anchor_width1_long30_joint_short30.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp40_rmt_gca_anchor_width1_long30_joint_debug.tar',
            '/content/drive/MyDrive/EcoCAR/training_runs/exp40_rmt_gca_anchor_width1_long30_joint.tar',
        ],
    },
]
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

for item in EVAL_ITEMS:
    config = item['config']
    run_tar = next((p for p in item['candidates'] if os.path.exists(p)), None)
    if run_tar is None:
        print('skip missing run tar candidates for config:', config, flush=True)
        for p in item['candidates']:
            print('  missing:', p, flush=True)
        continue
    cmd = [
        sys.executable, '-u', 'stage2/scripts/evaluate_joint_model.py',
        '--run-tar', run_tar,
        '--config', config,
        '--curve-tar', CURVE_TAR,
        '--curve-root', CURVE_ROOT,
        '--force-extract',
        '--limit-val', '1000',
    ]
    print('Evaluating:', run_tar, flush=True)
    try:
        run_streaming(cmd, log_path=os.path.join(LOG_DIR, f'{Path(run_tar).stem}_eval.log'))
    except subprocess.CalledProcessError as exc:
        print('eval failed for', run_tar, 'rc=', exc.returncode, flush=True)
        traceback.print_exc()
        print('--- continuing to next eval item ---', flush=True)
    except Exception as exc:
        print('unexpected eval error for', run_tar, ':', exc, flush=True)
        traceback.print_exc()
        print('--- continuing to next eval item ---', flush=True)


## Plot per-epoch trends from saved metrics JSON files
This cell is for the exact trend check we need before changing architecture again. It plots loss components, lane point MAE, matched LineIoU, existence F1, and false positive/false negative lane slots.


In [ ]:
import os, sys
METRICS = [
    '/content/drive/MyDrive/EcoCAR/training_runs/exp01_rmt_shared_joint_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp02_rmt_gca_joint_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp03_rmt_gca_lane_detail_joint_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp04_rmt_gca_lane_matching_joint_short10_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp05_rmt_gca_lane_detail_matching_joint_short10_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp07_rmt_gca_clrkd_lane_head_joint_short10_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp08_rmt_gca_clrkd_asl_cls_rescue_joint_short10_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp09_rmt_gca_clrkd_ohem_prior_encoder_joint_short10_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp10_rmt_gca_clrkd_separate_cls_path_joint_short10_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp11_rmt_gca_clrkd_iou_regression_joint_short10_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp12_rmt_gca_clrkd_iou_qfl_joint_short10_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp13_rmt_gca_clrkd_iou_sqrt_target_joint_short10_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp14_rmt_gca_clrkd_decoded_eval_joint_short10_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp15_rmt_gca_clrkd_oracle_diagnostic_joint_short10_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp16_rmt_gca_lane_query_head_joint_short10_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp17_rmt_gca_hybrid_prior_query_joint_short10_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp18_rmt_gca_query_anchor_dab_joint_short10_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp19_rmt_gca_bezier_query_joint_short10_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp20_rmt_gca_hybrid_with_stage1_aux_joint_short10_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp21_rmt_gca_lane_only_diagnostic_joint_short10_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp22_rmt_gca_grid_det_with_query_lane_joint_short10_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp23_rmt_gca_query_with_mask_aux_joint_short10_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp24_rmt_gca_hybrid_combined_signals_joint_short10_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp25_rmt_gca_mask_fixed_lambda_long_joint_short15_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp26_rmt_gca_mask_uncertainty_weighting_joint_short15_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp27_rmt_gca_mask_high_resolution_joint_short15_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp28_rmt_gca_mask_lane_dominant_joint_short15_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp29_rmt_gca_mask_uncertainty_long30_joint_long30_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp30_rmt_gca_mask_self_distillation_joint_short15_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp31_rmt_gca_mask_uncertainty_cosine_lr_joint_short20_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp32_rmt_gca_mask_self_distillation_cosine_lr_joint_short20_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp33_rmt_gca_mask_uncertainty_full_dataset_joint_full15_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp34_rmt_gca_anchor_clrkd_modern_training_joint_short20_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp35_rmt_gca_anchor_asl_amp_joint_short20_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp36_rmt_gca_anchor_iou_regression_joint_short20_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp37_rmt_gca_anchor_dual_score_joint_short20_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp38_rmt_gca_anchor_mask_consistency_joint_short20_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp39_rmt_gca_anchor_hungarian_joint_short20_metrics.json',
    '/content/drive/MyDrive/EcoCAR/training_runs/exp40_rmt_gca_anchor_width1_long30_joint_short30_metrics.json',
]
existing = [p for p in METRICS if os.path.exists(p)]
if not existing:
    print('No metrics JSON files found in the expected Drive paths yet.', flush=True)
else:
    cmd = [sys.executable, '-u', 'stage2/scripts/plot_stage2_metrics.py', '--metrics', *existing, '--out-dir', '/content/drive/MyDrive/EcoCAR/stage2/trend_plots']
    run_streaming(cmd, log_path=os.path.join(LOG_DIR, 'stage2_metric_plotting.log'))


## Optional video profiling with drawn detection boxes and lane curves
Set `VIDEO_PATH` to a real Drive video. This cell now draws both vehicle boxes and lane curves into `profile_preview.mp4`; it also prints per-frame detection/lane counts.


In [ ]:
from pathlib import Path
import os, sys

VIDEO_PATH = '/content/drive/MyDrive/EcoCAR/video/test.mp4'

# Prefer the newest stable Exp2G run; fall back to Exp2E. Skip if neither exists.
PROFILE_CANDIDATES = [
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp38_rmt_gca_anchor_mask_consistency_joint_short20.tar', 'stage2/configs/exp38_rmt_gca_anchor_mask_consistency_joint.yaml', 'video_profile_exp38'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp39_rmt_gca_anchor_hungarian_joint_short20.tar', 'stage2/configs/exp39_rmt_gca_anchor_hungarian_joint.yaml', 'video_profile_exp39'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp40_rmt_gca_anchor_width1_long30_joint_short30.tar', 'stage2/configs/exp40_rmt_gca_anchor_width1_long30_joint.yaml', 'video_profile_exp40'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp37_rmt_gca_anchor_dual_score_joint_short20.tar', 'stage2/configs/exp37_rmt_gca_anchor_dual_score_joint.yaml', 'video_profile_exp37'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp36_rmt_gca_anchor_iou_regression_joint_short20.tar', 'stage2/configs/exp36_rmt_gca_anchor_iou_regression_joint.yaml', 'video_profile_exp36'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp12_rmt_gca_clrkd_iou_qfl_joint_short10.tar', 'stage2/configs/exp12_rmt_gca_clrkd_iou_qfl_joint.yaml', 'video_profile_exp12'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp17_rmt_gca_hybrid_prior_query_joint_short10.tar', 'stage2/configs/exp17_rmt_gca_hybrid_prior_query_joint.yaml', 'video_profile_exp17'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp19_rmt_gca_bezier_query_joint_short10.tar', 'stage2/configs/exp19_rmt_gca_bezier_query_joint.yaml', 'video_profile_exp19'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp21_rmt_gca_lane_only_diagnostic_joint_short10.tar', 'stage2/configs/exp21_rmt_gca_lane_only_diagnostic_joint.yaml', 'video_profile_exp21'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp23_rmt_gca_query_with_mask_aux_joint_short10.tar', 'stage2/configs/exp23_rmt_gca_query_with_mask_aux_joint.yaml', 'video_profile_exp23'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp25_rmt_gca_mask_fixed_lambda_long_joint_short15.tar', 'stage2/configs/exp25_rmt_gca_mask_fixed_lambda_long_joint.yaml', 'video_profile_exp25'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp26_rmt_gca_mask_uncertainty_weighting_joint_short15.tar', 'stage2/configs/exp26_rmt_gca_mask_uncertainty_weighting_joint.yaml', 'video_profile_exp26'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp28_rmt_gca_mask_lane_dominant_joint_short15.tar', 'stage2/configs/exp28_rmt_gca_mask_lane_dominant_joint.yaml', 'video_profile_exp28'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp29_rmt_gca_mask_uncertainty_long30_joint_long30.tar', 'stage2/configs/exp29_rmt_gca_mask_uncertainty_long30_joint.yaml', 'video_profile_exp29'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp31_rmt_gca_mask_uncertainty_cosine_lr_joint_short20.tar', 'stage2/configs/exp31_rmt_gca_mask_uncertainty_cosine_lr_joint.yaml', 'video_profile_exp31'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp33_rmt_gca_mask_uncertainty_full_dataset_joint_full15.tar', 'stage2/configs/exp33_rmt_gca_mask_uncertainty_full_dataset_joint.yaml', 'video_profile_exp33'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp35_rmt_gca_anchor_asl_amp_joint_short20.tar', 'stage2/configs/exp35_rmt_gca_anchor_asl_amp_joint.yaml', 'video_profile_exp35'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp34_rmt_gca_anchor_clrkd_modern_training_joint_short20.tar', 'stage2/configs/exp34_rmt_gca_anchor_clrkd_modern_training_joint.yaml', 'video_profile_exp34'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp32_rmt_gca_mask_self_distillation_cosine_lr_joint_short20.tar', 'stage2/configs/exp32_rmt_gca_mask_self_distillation_cosine_lr_joint.yaml', 'video_profile_exp32'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp30_rmt_gca_mask_self_distillation_joint_short15.tar', 'stage2/configs/exp30_rmt_gca_mask_self_distillation_joint.yaml', 'video_profile_exp30'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp27_rmt_gca_mask_high_resolution_joint_short15.tar', 'stage2/configs/exp27_rmt_gca_mask_high_resolution_joint.yaml', 'video_profile_exp27'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp24_rmt_gca_hybrid_combined_signals_joint_short10.tar', 'stage2/configs/exp24_rmt_gca_hybrid_combined_signals_joint.yaml', 'video_profile_exp24'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp22_rmt_gca_grid_det_with_query_lane_joint_short10.tar', 'stage2/configs/exp22_rmt_gca_grid_det_with_query_lane_joint.yaml', 'video_profile_exp22'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp20_rmt_gca_hybrid_with_stage1_aux_joint_short10.tar', 'stage2/configs/exp20_rmt_gca_hybrid_with_stage1_aux_joint.yaml', 'video_profile_exp20'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp18_rmt_gca_query_anchor_dab_joint_short10.tar', 'stage2/configs/exp18_rmt_gca_query_anchor_dab_joint.yaml', 'video_profile_exp18'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp16_rmt_gca_lane_query_head_joint_short10.tar', 'stage2/configs/exp16_rmt_gca_lane_query_head_joint.yaml', 'video_profile_exp16'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp15_rmt_gca_clrkd_oracle_diagnostic_joint_short10.tar', 'stage2/configs/exp15_rmt_gca_clrkd_oracle_diagnostic_joint.yaml', 'video_profile_exp15'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp14_rmt_gca_clrkd_decoded_eval_joint_short10.tar', 'stage2/configs/exp14_rmt_gca_clrkd_decoded_eval_joint.yaml', 'video_profile_exp14'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp13_rmt_gca_clrkd_iou_sqrt_target_joint_short10.tar', 'stage2/configs/exp13_rmt_gca_clrkd_iou_sqrt_target_joint.yaml', 'video_profile_exp13'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp11_rmt_gca_clrkd_iou_regression_joint_short10.tar', 'stage2/configs/exp11_rmt_gca_clrkd_iou_regression_joint.yaml', 'video_profile_exp11'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp10_rmt_gca_clrkd_separate_cls_path_joint_short10.tar', 'stage2/configs/exp10_rmt_gca_clrkd_separate_cls_path_joint.yaml', 'video_profile_exp10'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp09_rmt_gca_clrkd_ohem_prior_encoder_joint_short10.tar', 'stage2/configs/exp09_rmt_gca_clrkd_ohem_prior_encoder_joint.yaml', 'video_profile_exp09'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp08_rmt_gca_clrkd_asl_cls_rescue_joint_short10.tar', 'stage2/configs/exp08_rmt_gca_clrkd_asl_cls_rescue_joint.yaml', 'video_profile_exp08'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp07_rmt_gca_clrkd_lane_head_joint_short10.tar', 'stage2/configs/exp07_rmt_gca_clrkd_lane_head_joint.yaml', 'video_profile_exp07'),
    ('/content/drive/MyDrive/EcoCAR/training_runs/exp04_rmt_gca_lane_matching_joint_short10.tar',  'stage2/configs/exp04_rmt_gca_lane_matching_joint.yaml',  'video_profile_exp04'),
]
PROFILE_RUN_TAR = None
PROFILE_CONFIG = None
PROFILE_OUT = None
for tar, cfg, out_tag in PROFILE_CANDIDATES:
    if os.path.exists(tar):
        PROFILE_RUN_TAR = tar
        PROFILE_CONFIG = cfg
        PROFILE_OUT = f'/content/drive/MyDrive/EcoCAR/stage2/{out_tag}'
        break

if not os.path.exists(VIDEO_PATH):
    print('Skip video profile because VIDEO_PATH does not exist:', VIDEO_PATH, flush=True)
elif PROFILE_RUN_TAR is None:
    print('Skip video profile: no candidate run tar exists.', flush=True)
    for tar, _, _ in PROFILE_CANDIDATES:
        print('  missing:', tar, flush=True)
else:
    print('Profiling with run_tar:', PROFILE_RUN_TAR, flush=True)
    print('               config:', PROFILE_CONFIG, flush=True)
    cmd = [
        sys.executable, '-u', 'stage2/scripts/profile_joint_video.py',
        '--config', PROFILE_CONFIG,
        '--run-tar', PROFILE_RUN_TAR,
        '--video', VIDEO_PATH,
        '--output-dir', PROFILE_OUT,
        '--write-video',
        '--max-frames', '600',
        '--det-conf', '0.05',
        '--det-iou', '0.60',
        '--lane-conf', '0.30',
    ]
    run_streaming(cmd, log_path=os.path.join(LOG_DIR, f'{Path(PROFILE_RUN_TAR).stem}_video_profile.log'))
